# MSIRI/MCIA — Sentinel-2 Cloud Reconstruction: Baseline U-Net (Kaggle)

Trains the reference-conditioned U-Net baseline on the synthetic supervised dataset.

**Before running:** Notebook settings (right sidebar) -> **Accelerator: GPU T4 x2** (or P100/L4), **Internet: On**, and **Add Input** -> your `synthetic_dataset` Kaggle Dataset.

Run the cells top to bottom. Cell 4 (fast-dev) and Cell 5 (sanity) take ~1 minute each and must pass before you start the full run in Cell 6.

## 1. Get the code and install the one extra dependency
Edit `REPO_URL` to your GitHub repo.

In [ ]:
REPO_URL = "https://github.com/<you>/<repo>.git"   # <-- your repo
!git clone -q $REPO_URL /kaggle/working/trial3
%cd /kaggle/working/trial3
!pip install -q rasterio      # needed by the dataset import chain; torch/pandas/tb are preinstalled
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. Locate the dataset (fast, no slug needed)
Walks **directories only** and prunes the huge `patch_library` / `cloud_tile_library` / split folders, so it finds `samples/train` in seconds instead of enumerating the ~1M tiny files. (Do NOT use a recursive `**` glob here — it would scan the whole dataset and hang.)

In [ ]:
import os
print("mounted inputs:", os.listdir("/kaggle/input"))

def find_dataset_root(base="/kaggle/input"):
    for root, dirs, _files in os.walk(base):
        dirs[:] = [d for d in dirs
                   if d not in ("patch_library", "cloud_tile_library",
                                "train", "validation", "test")]
        if "samples" in dirs and os.path.isdir(os.path.join(root, "samples", "train")):
            return root
    return None

DATA_ROOT = find_dataset_root()
assert DATA_ROOT, "Could not find samples/train under /kaggle/input"
print("DATA_ROOT =", DATA_ROOT)
print("contents:", sorted(os.listdir(DATA_ROOT)))   # expect: patch_library cloud_tile_library samples
with os.scandir(f"{DATA_ROOT}/samples/train") as it:
    print("train samples:", sum(1 for e in it if e.name.endswith(".json")))

## 3. Point the config at the dataset
Sets **only** the indented `data.root:` line (not `output_root:`), and keeps `output_root: experiments` so outputs go to the writable `/kaggle/working` (never the read-only `/kaggle/input`).

In [ ]:
import re, pathlib
p = pathlib.Path("configs/unet_baseline.yaml")
text = p.read_text()
text = re.sub(r"(?m)^output_root:.*", "output_root: experiments", text)
text = re.sub(r"(?m)^\s+root:.*", f"  root: {DATA_ROOT}", text, count=1)
p.write_text(text)
print([l for l in text.splitlines() if "root:" in l])

## 4. Fast-dev dry run (~30 s) — exercises the WHOLE pipeline on a few batches
Confirms data loading, model, loss, metrics, checkpointing, visualisation and the summary all work before you spend GPU hours.

In [ ]:
!python run_train.py --config configs/unet_baseline.yaml --limit-batches 3 --epochs 1 --name dryrun

## 5. Sanity check (~1 min) — proves the model actually learns
Overfits 64 samples for 2 epochs and asserts the loss drops. Prints `SANITY PASSED/FAILED`.

In [ ]:
!python run_train.py --config configs/unet_baseline.yaml --sanity

## 6. Full training
Outputs -> `/kaggle/working/experiments/unet_baseline/` (checkpoints, metrics.csv, visualizations, tensorboard, model_summary.json). `resume: auto` is set, so re-running this cell after an interruption continues from the last epoch.

In [ ]:
!python run_train.py --config configs/unet_baseline.yaml

## 7. Monitor (run in parallel / after)

In [ ]:
import pandas as pd
run = "/kaggle/working/experiments/unet_baseline"
print(open(f"{run}/model_summary.json").read())
display(pd.read_csv(f"{run}/metrics.csv").tail())
%load_ext tensorboard
%tensorboard --logdir {run}/tensorboard

## 8. Resume after a Kaggle session ends
1. **Save Version** (commit) so `/kaggle/working` is persisted as this notebook's output.
2. New session -> **Add Input -> your own notebook's previous output**, then re-run Cells 1-3 and Cell 6. `resume: auto` finds `latest.pt` (in `/kaggle/working`, or under `/kaggle/input` if re-attached) and continues.

## 9. Evaluate the best checkpoint + save the model

In [ ]:
CKPT = "/kaggle/working/experiments/unet_baseline/checkpoints/best.pt"
!python run_eval.py --checkpoint {CKPT} --split test --out /kaggle/working/eval_test.json
print(open("/kaggle/working/eval_test.json").read())
# Then 'Save Version' and download best.pt + reports from the notebook's Output tab.